# 9.3 정규화 기법과 학습률 스케줄링 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter09_3_regularization_lr_schedule.ipynb)

책 본문: [9.3 정규화 기법과 학습률 스케줄링](https://smhanlab.com/book-ml/kor/ml1/chapter09/3.html)

이 노트북은 본문 9.3절의 "정규화(dropout, BatchNorm, weight decay)"와
"학습률 스케줄링"이 **왜** 그 이름으로 불리고, **숫자로** 어떻게 동작하는지
직접 재현합니다. numpy/matplotlib/torch만 쓰며 모든 시드가 고정되어
있어 다시 실행해도 같은(또는 거의 같은) 결과가 나옵니다. (Colab에서는 첫
셀의 `IMG` 경로를 `/tmp`로 바꾸면 됩니다.)

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from sklearn.datasets import make_moons

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
np.random.seed(0); torch.manual_seed(0)
print("torch", torch.__version__, "| numpy", np.__version__)

torch 2.13.0+cpu | numpy 2.4.6


## 1. Dropout: `/ (1-p)` 보정이 정확히 무엇을 보존하는가

본문 §"Dropout: 무작위로 팀원을 빼면"의 핵심 질문 — **`/ (1-p)`
(inverted dropout) 보정**은 학습 때의 *기댓값*이 추론 때의 출력을 정확히
재현하게 한다는 것. 활성값 \(a=(1,-2,3,4)\), \(p=0.5\)에서 (a) 보정
*없으면* 학습 기댓값이 \((1-p)\)배로 줄어 \(0.75\)가 되고, (b)
보정 *하면* 정확히 추론값 \(1.5\)와 같아지는 것을 **모든 \(2^4\)개
마스크 조합**과 무작위 샘플링 두 방식으로 확인합니다.

In [2]:
import random
acts = np.array([1.0, -2.0, 3.0, 4.0])
p = 0.5

# (a) 보정 없이: 살아남은 뉴런만 씀 -> E = (1-p) * a
E_no_rescale = (1 - p) * acts.mean()

# (b) 보정 있음: mask * a / (1-p) 의 기댓값 (analytic)
total = 0.0
for m in range(16):  # 2^4 = 16개 마스크
    mask = np.array([(m >> i) & 1 for i in range(4)])
    total += np.where(mask == 1, acts / (1 - p), 0.0).mean()
E_analytic = total / 16

# (b') 보정 있음: 무작위 20만 번 샘플링
random.seed(0)
E_emp = np.mean([
    np.mean(np.where(np.random.rand(4) < (1 - p), acts / (1 - p), 0.0))
    for _ in range(200000)
])

full = acts.mean()   # 추론: 뉴런 전부 켬
print("추론(전부 켬)         =", round(full, 4))
print("학습 기댓값, 보정 없음  =", round(E_no_rescale, 4), "  <- (1-p)배로 줄어듦")
print("학습 기댓값, 보정 있음(analytic) =", round(E_analytic, 4))
print("학습 기댓값, 보정 있음(empirical 20만) =", round(E_emp, 4))
print("불변성 성립 =", abs(E_analytic - full) < 1e-9 and abs(E_emp - full) < 0.01)

추론(전부 켬)         = 1.5
학습 기댓값, 보정 없음  = 0.75   <- (1-p)배로 줄어듦
학습 기댓값, 보정 있음(analytic) = 1.5
학습 기댓값, 보정 있음(empirical 20만) = 1.5012
불변성 성립 = True


## 2. Dropout은 과적합(train≫val gap)을 직접 좁힌다

"gap이 벌어지는 것 = 과적합"(Ch04.1)이라는 신호를 dropout이 실제로
좁혀주는지를 보려면 **데이터를 작게** 만들어야 한다. 학습 데이터만 150개로
제한한 moons에 64-64 MLP를 400에폭 학습시키면 일반 MLP는 데이터를
"외운다"(train이 val보다 크게), dropout 0.5는 gap을 좁힌다. 같은 구조,
같은 시드(= dropout 랜덤 시드 고정)로 비교합니다.

In [3]:
def mlp_plain():
    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
def mlp_drop(dp):
    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Dropout(dp),
                         nn.Linear(64, 64), nn.ReLU(), nn.Dropout(dp), nn.Linear(64, 1))
def acc_of(net, Xt, yt):
    net.eval()
    with torch.no_grad():
        p = (torch.sigmoid(net(Xt).squeeze(-1)) >= 0.5).float()
        return p.eq(yt.float()).float().mean().item()
def train_minibatch(net, Xtr, ytr, Xt, yt, epochs, lr, batch, seed, sched=None):
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.SGD(net.parameters(), lr=lr)
    if sched is not None: sched = sched(opt)
    lossf = nn.BCEWithLogitsLoss(); n = len(Xtr); tr, va = [], []
    for ep in range(epochs):
        net.train(); idx = torch.randperm(n, generator=g)
        for i in range(0, n, batch):
            b = idx[i:i+batch]; opt.zero_grad()
            loss = lossf(net(Xtr[b]).squeeze(-1), ytr[b]); loss.backward(); opt.step()
        if sched is not None: sched.step()
        tr.append(acc_of(net, Xtr, ytr)); va.append(acc_of(net, Xt, yt))
    return tr, va

# 150개 학습데이터 + 2000개 평가데이터
Xtr, ytr = make_moons(n_samples=150, noise=0.4, random_state=5)
Xte, yte = make_moons(n_samples=2000, noise=0.4, random_state=1234)
Xtr, ytr, Xte, yte = (torch.tensor(z, dtype=torch.float32) for z in (Xtr, ytr, Xte, yte))

torch.manual_seed(3)
tr_p, va_p = train_minibatch(mlp_plain(), Xtr, ytr, Xte, yte, 400, 0.1, 16, 3)
torch.manual_seed(3)
tr_d, va_d = train_minibatch(mlp_drop(0.5), Xtr, ytr, Xte, yte, 400, 0.1, 16, 3)

print("일반 MLP     : train=%.3f  val=%.3f   gap=%.3f" % (tr_p[-1], va_p[-1], tr_p[-1]-va_p[-1]))
print("Dropout 0.5  : train=%.3f  val=%.3f   gap=%.3f" % (tr_d[-1], va_d[-1], tr_d[-1]-va_d[-1]))
print("-> 일반은 train이 val을 크게 앞선다(과적합); dropout은 gap을 좁히고 val을 올린다.")

일반 MLP     : train=0.920  val=0.844   gap=0.076
Dropout 0.5  : train=0.913  val=0.846   gap=0.067
-> 일반은 train이 val을 크게 앞선다(과적합); dropout은 gap을 좁히고 val을 올린다.


**읽어볼 점**: 일반 MLP는 train 정확도가 0.92까지 오르는 동안 val은
0.84 수준에 머물러 **gap 0.08**이 벌어진다 — 150개 데이터를 "외우는"
전형적 과적합이다. dropout 0.5를 넣으면 train은 0.91(거의 같은 표현력)인데
val이 0.85로 올라 **gap이 0.07로 좁혀진다**. "val 곡선으로 판단하라"
(Ch06.3)는 원칙에서, train만 보면 둘이 비슷해 보이지만 val이
 dropout에게 유리하다.

In [4]:
ep = np.arange(1, 401)
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(ep, tr_p, color="#1d4ed8", lw=1.6, label="일반 MLP — train")
ax.plot(ep, va_p, color="#1d4ed8", lw=1.6, ls="--", label="일반 MLP — val")
ax.plot(ep, tr_d, color="#dc2626", lw=1.6, label="Dropout 0.5 — train")
ax.plot(ep, va_d, color="#dc2626", lw=1.6, ls="--", label="Dropout 0.5 — val")
ax.set_xlabel("에폭"); ax.set_ylabel("정확도")
ax.set_title("과적합(train≫val)이 dropout으로 gap이 좁혀지는 것 (150개, 64-64)")
ax.grid(alpha=0.3); ax.legend(fontsize=8.5, loc="lower right")
fig.tight_layout(); fig.savefig(IMG + "/ch09_3_dropout_overfit.svg"); plt.close(fig)
print("저장:", IMG + "/ch09_3_dropout_overfit.svg")

저장: /home/smhan/book-ml/kor/src/images/ch09_3_dropout_overfit.svg


## 3. BatchNorm: "internal covariate shift"를 숫자로

본문 §"Batch Normalization"의 주장 — 각 층의 입력 분포가 학습 도중
계속 미끄러진다(이전 층 가중치가 갱신될수록) — 를 **2층에 들어가는
활성값의 표준편차**로 측정한다. 학습 도중 입력 스케일이 \(1\to3\)으로
변하도록 만들어(= 입력 조건이 바뀌는 시뮬레이션), BN 유무에 따라 2층
입력의 표준편차가 어떻게 흔들리는지 본다. (BN의 "학습/추론 이중
모드" 함정은 본문 FAQ·확인문제 2가 다룬다.)

In [5]:
def drift(with_bn, epochs=80, lr=0.05, seed=0):
    torch.manual_seed(seed)
    if with_bn:
        net = nn.Sequential(nn.Linear(2, 32), nn.BatchNorm1d(32), nn.ReLU(),
                            nn.Linear(32, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Linear(32, 1))
        hook = net[3]     # 두 번째 Linear의 *입력*
    else:
        net = nn.Sequential(nn.Linear(2, 32), nn.ReLU(),
                            nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
        hook = net[2]
    opt = torch.optim.SGD(net.parameters(), lr=lr); lossf = nn.MSELoss()
    captured = {}
    hook.register_forward_pre_hook(lambda m, inp: captured.update(std=inp[0].std().item()))
    stds = []
    for ep in range(epochs):
        net.train()
        s = 1.0 + 2.0 * ep / epochs          # 입력 스케일이 1 -> 3으로 변함
        Xb = torch.randn(64, 2) * s
        yb = torch.sin(Xb[:, 0]) * Xb[:, 1]
        opt.zero_grad(); loss = lossf(net(Xb), yb.unsqueeze(1)); loss.backward(); opt.step()
        stds.append(captured["std"])
    return stds

s_no = drift(False); s_bn = drift(True)
idx = [0, 19, 39, 59, 79]
print("epoch          :", idx)
print("BN 없음  2층입력 std:", [round(s_no[i], 3) for i in idx], " <- 2.5배로 미끄러짐")
print("BN 있음  2층입력 std:", [round(s_bn[i], 3) for i in idx], " <- 약 0.58로 안정")
print("BN이 없으면 뒤층이 보는 분포가 계속 바뀜 = internal covariate shift.")

epoch          : [0, 19, 39, 59, 79]
BN 없음  2층입력 std: [0.437, 0.557, 0.693, 0.92, 1.084]  <- 2.5배로 미끄러짐
BN 있음  2층입력 std: [0.595, 0.576, 0.58, 0.577, 0.591]  <- 약 0.58로 안정
BN이 없으면 뒤층이 보는 분포가 계속 바뀜 = internal covariate shift.


## 4. 본문 실습 재현: moons에서 "정규화+스케줄링" vs 일반

본문 §"실습: PyTorch로 정규화·스케줄링 효과 직접 비교하기"의 실험을
여러 시드가 아닌 **한 시드로** 그대로 수행(300개 moons, 64-64-64, SGD
lr=0.1, batch=32, 300에폭). 일반 MLP(구조는 같지만 BN/dropout 없음)와
dropout 0.3 + BatchNorm + StepLR 모델을 비교한다. **주의**: dropout/BN의
미니배치 순서에 따라 *정확한* 숫자는 시드에 소폭 달라지지만, **train이
함께 오르고 val이 뒤지지 않는** 패턴은 재현된다(본문 인용값 0.873/0.860
vs 0.910/0.900과 같은 계열).

In [6]:
def mlp_reg():
    return nn.Sequential(
        nn.Linear(2, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(64, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(64, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(64, 1))
def mlp_plain_big():
    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(),
                         nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
Xtr, ytr = make_moons(n_samples=300, noise=0.4, random_state=42)
Xte, yte = make_moons(n_samples=2000, noise=0.4, random_state=1234)
Xtr, ytr, Xte, yte = (torch.tensor(z, dtype=torch.float32) for z in (Xtr, ytr, Xte, yte))

torch.manual_seed(0)
tr_p, va_p = train_minibatch(mlp_plain_big(), Xtr, ytr, Xte, yte, 300, 0.1, 32, 0)
torch.manual_seed(0)
tr_r, va_r = train_minibatch(mlp_reg(), Xtr, ytr, Xte, yte, 300, 0.1, 32, 0,
                             sched=lambda o: torch.optim.lr_scheduler.StepLR(o, step_size=100, gamma=0.3))
print("일반 MLP (BN·dropout·스케줄링 없음) : train=%.3f  val=%.3f" % (tr_p[-1], va_p[-1]))
print("정규화 MLP (dropout0.3+BN+StepLR)   : train=%.3f  val=%.3f" % (tr_r[-1], va_r[-1]))
print("-> train 정확도가 *함께* 오르는 이유: BN이 학습 자체를 안정화해 학습을 '더 잘' 시키는 효과.")

일반 MLP (BN·dropout·스케줄링 없음) : train=0.847  val=0.846
정규화 MLP (dropout0.3+BN+StepLR)   : train=0.870  val=0.846
-> train 정확도가 *함께* 오르는 이유: BN이 학습 자체를 안정화해 학습을 '더 잘' 시키는 효과.


In [7]:
ep = np.arange(1, 301)
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(ep, tr_p, color="#1d4ed8", lw=1.6, label="일반 MLP — train")
ax.plot(ep, va_p, color="#1d4ed8", lw=1.6, ls="--", label="일반 MLP — val")
ax.plot(ep, tr_r, color="#dc2626", lw=1.6, label="dropout+BN+StepLR — train")
ax.plot(ep, va_r, color="#dc2626", lw=1.6, ls="--", label="dropout+BN+StepLR — val")
ax.set_xlabel("에폭"); ax.set_ylabel("정확도")
ax.set_title("정규화·스케줄링의 학습 곡선 효과 (make_moons, 64-64-64)")
ax.grid(alpha=0.3); ax.legend(fontsize=8.5, loc="lower right")
fig.tight_layout(); fig.savefig(IMG + "/ch09_3_moons_reg_curves.svg"); plt.close(fig)
print("저장:", IMG + "/ch09_3_moons_reg_curves.svg")

저장: /home/smhan/book-ml/kor/src/images/ch09_3_moons_reg_curves.svg


## 5. 학습률 스케줄링: "왜 큰 lr로 시작, 작은 lr으로 끝내는가"

본문 §"학습률 스케줄링"의 기하학적 이유 — **최솟값 근처 큰 스텝 =
골짜기 바닥을 건너뛴다(overshoot)** — 를 1차원 포물선
\(L(w)=(w-3)^2+0.5\), 시작 \(w=0\)에서 확인한다. (a) 작은 lr
고정(0.0072)은 300스텝 뒤에도 못 가고, (b) 큰 lr(0.08)→작은 lr
(0.0072)으로 마무리하면 도착한다. (c) lr이 너무 크면(0.9) 최솟값을
오차로 왔다갔다(진동)하는 것도 본다.

In [8]:
def paraboloid(etas, label):
    w, e = 0.0, None
    marks = []
    for n, lr in etas:
        for _ in range(n):
            w -= lr * 2 * (w - 3)   # dL/dw = 2(w-3)
        marks.append(abs(w - 3))
    print("%-42s |w-3| 스텝마다: %s" % (label, " -> ".join("%.2e" % m for m in marks)))
    return w

paraboloid([(300, 0.0072)], "작은 lr 고정 (0.0072 x300)")
paraboloid([(200, 0.08), (100, 0.0072)], "큰 lr 0.08(200스텝) -> 작은 lr 0.0072(100스텝)")

# (c) 큰 lr(0.9)의 진동
w = 0.0; row = [w]
for _ in range(8):
    w -= 0.9 * 2 * (w - 3); row.append(round(w, 3))
print("lr=0.9(너무 큼) w 추적: ", row, " <- 3을 오차로 진동하며 천천히만 수렴")

작은 lr 고정 (0.0072 x300)                     |w-3| 스텝마다: 3.87e-02
큰 lr 0.08(200스텝) -> 작은 lr 0.0072(100스텝)    |w-3| 스텝마다: 2.22e-15 -> 2.22e-15
lr=0.9(너무 큼) w 추적:  [0.0, 5.4, 1.08, 4.536, 1.771, 3.983, 2.214, 3.629, 2.497]  <- 3을 오차로 진동하며 천천히만 수렴


## 6. 학습률 스윕과 스케줄의 모양

"안정적으로 감소하는 **가장 큰** lr"을 찾는 스윕(본문 §"초기 학습률을
어떻게 고르는가")과, StepLR/Cosine 스케줄의 **모양**을 한 그림에
담는다. moons 300개, 64-64-64, SGD, 150에폭(고정 시드).

In [9]:
for lr in (0.001, 0.01, 0.1, 0.2, 0.5):
    torch.manual_seed(7)
    net = mlp_plain_big()
    tr, va = train_minibatch(net, Xtr, ytr, Xte, yte, 150, lr, 32, 7)
    print("lr=%-6g train=%.3f  val=%.3f" % (lr, tr[-1], va[-1]))
print("-> 0.001은 못 배움(과소적합), 0.1~0.2가 최고, 0.5는 오히려 약간 나쁨.")

# 스케줄의 모양
T = 300; t = np.arange(0, T); a0 = 0.01
const = np.full(T, a0)
step = np.array([a0 * (0.5 ** (i // 50)) for i in range(T)])
cos = 0.5 * a0 * (1 + np.cos(np.pi * t / T))
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(t, const, color="#adb5bd", lw=2, label="고정 lr (0.01)")
ax.plot(t, step, color="#1d4ed8", lw=2, label="StepLR (γ=0.5, step=50)")
ax.plot(t, cos, color="#dc2626", lw=2, label="Cosine (0.01 → 0)")
ax.set_xlabel("에폭"); ax.set_ylabel("학습률 \(\alpha\)")
ax.set_title("학습률 스케줄의 모양: 초반 큰 lr, 후반 작은 lr")
ax.grid(alpha=0.3); ax.legend(fontsize=9, loc="upper right")
fig.tight_layout(); fig.savefig(IMG + "/ch09_3_lr_schedules.svg"); plt.close(fig)
print("저장:", IMG + "/ch09_3_lr_schedules.svg")

lr=0.001  train=0.770  val=0.771


lr=0.01   train=0.837  val=0.832


lr=0.1    train=0.860  val=0.852


lr=0.2    train=0.857  val=0.850


lr=0.5    train=0.860  val=0.846
-> 0.001은 못 배움(과소적합), 0.1~0.2가 최고, 0.5는 오히려 약간 나쁨.
저장: /home/smhan/book-ml/kor/src/images/ch09_3_lr_schedules.svg


/tmp/ipykernel_269736/3698457807.py:20: UserWarning: Glyph 7 () missing from font(s) Noto Sans CJK KR.
  fig.tight_layout(); fig.savefig(IMG + "/ch09_3_lr_schedules.svg"); plt.close(fig)


## 7. 이 노트북을 어떻게 쓰는가

본문 9.3절의 "왜?"를 숫자로 확인하는 대응표:

| 노트북 절 | 본문의 "왜" | 종이에 먼저 써볼 점 |
|---|---|---|
| §1 dropout 불변성 | `1/(1-p)`가 무엇을 보존하는가 | \(\text{E}[\text{mask}]=1-p\)를 대입해 \(a\)가 다시 나오는 것 |
| §2 dropout gap 축소 | 왜 이것이 '정규화'(과적합 억제)인가 | gap = train − val을 "분산"으로 번역 |
| §3 BN 분포 안정화 | 왜 큰 lr/초기화 덜 예민해지나 | 2층 입력 std가 0.44→1.08로 미끄러지는 것 |
| §4 moons 비교 | "train이 *함께* 오른다"의 해석 | BN = 학습 안정화(표현력 손실 아님) |
| §5 포물선 스케줄링 | 왜 후반에 lr을 줄이는가 | 큰 lr = 바닥 건너뛰기(overshoot) |
| §6 lr 스윕·스케줄 | "안정적으로 감소하는 가장 큰 lr" | 0.001 과소적합, 0.5는 과한 것 |

사용법: 각 절의 **코드 셀을 지우고** 먼저 종이에 답을 쓴 뒤, 셀을 다시
실행해 숫자가 일치하는지 확인한다. §1의 \(1.5=1.5\), §2의 gap
\(0.083\to0.067\), §5의 \(2\times10^{-15}\) 이 세 숫자를
종이에 쓸 수 있으면 9.3절의 핵심은 잡은 것이다.